# DinoV2 Dogs-vs-Cats Hypothesis: Does Data Augmentation Help? (Parallel 2xT4)

Repository purpose: reproducible, config-driven DINOv2 experiments for Kaggle Dogs-vs-Cats.

Hypothesis in this notebook: stronger train-time augmentation can improve validation quality vs a near-no-augmentation baseline.

Flow: bootstrap -> config variants -> preprocess -> sanity -> train x2 -> compare -> review artifacts.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')

REPO_URL = 'https://github.com/mruniverse8/kaggle-experiments-.git'
REPO_DIR = Path('/kaggle/working/kaggle-experiments-')
BRANCH = 'dogs_vs_cats'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', '--all'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

current_branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip()
current_commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
print('Git branch:', current_branch)
print('Git commit:', current_commit)
print('Repo ready at:', REPO_DIR)


In [ ]:
import os
import json
import copy
from pathlib import Path

BASE_CFG_PATH = os.environ.get('BASE_CFG_PATH', 'dogs_vs_cats/configs/experiments/dinov2_vitb14_parallel_t4x2_small.json')
PATHS_CFG = os.environ.get('PATHS_CFG', 'dogs_vs_cats/configs/paths_kaggle.json')
AUG_COMPARE_TAG = os.environ.get('AUG_COMPARE_TAG', 'augcmp')

BASE_CFG = json.loads(Path(BASE_CFG_PATH).read_text())
PATHS = json.loads(Path(PATHS_CFG).read_text())

safe_tag = ''.join(ch if (ch.isalnum() or ch in ['_', '-']) else '_' for ch in AUG_COMPARE_TAG)
generated_cfg_dir = Path('dogs_vs_cats/configs/experiments/generated')
generated_cfg_dir.mkdir(parents=True, exist_ok=True)

profiles = [
    {
        'key': 'no_aug',
        'description': 'Near no augmentation baseline',
        'train_aug': {
            'random_resized_crop_scale': [1.0, 1.0],
            'horizontal_flip_p': 0.0,
            'color_jitter': [0.0, 0.0, 0.0, 0.0],
            'random_erasing_p': 0.0,
        },
    },
    {
        'key': 'strong_aug',
        'description': 'Stronger augmentation profile',
        'train_aug': {
            'random_resized_crop_scale': [0.5, 1.0],
            'horizontal_flip_p': 0.5,
            'color_jitter': [0.3, 0.3, 0.3, 0.08],
            'random_erasing_p': 0.25,
        },
    },
]

EXPERIMENTS = []
for profile in profiles:
    cfg = copy.deepcopy(BASE_CFG)
    exp_name = f"{BASE_CFG['experiment_name']}_{safe_tag}_{profile['key']}"
    cfg['experiment_name'] = exp_name
    cfg.setdefault('augmentation', {})
    cfg['augmentation'].setdefault('train', {})
    cfg['augmentation']['train'].update(profile['train_aug'])
    cfg['notes'] = (
        f"augmentation hypothesis run ({profile['key']}) derived from {BASE_CFG_PATH}"
    )
    cfg_path = generated_cfg_dir / f"{exp_name}.json"
    cfg_path.write_text(json.dumps(cfg, indent=2, sort_keys=True))
    EXPERIMENTS.append(
        {
            'profile_key': profile['key'],
            'description': profile['description'],
            'cfg_path': str(cfg_path),
            'experiment_name': exp_name,
        }
    )

PRIMARY_CFG_PATH = EXPERIMENTS[0]['cfg_path']

print('Using base experiment config:', BASE_CFG_PATH)
print('Using paths config:', PATHS_CFG)
print('Generated comparison configs:')
for exp in EXPERIMENTS:
    print('-', exp['profile_key'], '->', exp['cfg_path'])


In [ ]:
import sys
import subprocess
from pathlib import Path

manifest_dir = Path(PATHS['manifests_dir'])
required = [
    manifest_dir / 'train_manifest.csv',
    manifest_dir / 'val_manifest.csv',
    manifest_dir / 'test_manifest.csv',
]

if all(path.exists() for path in required):
    print('Preprocess skipped: manifests already exist')
else:
    subprocess.run([
        sys.executable,
        'dogs_vs_cats/src/preprocess_competition_data.py',
        '--paths-config', PATHS_CFG,
        '--experiment-config', PRIMARY_CFG_PATH,
    ], check=True)


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

for exp in EXPERIMENTS:
    print(f"\n[sanity] {exp['profile_key']} -> {exp['experiment_name']}")
    subprocess.run([
        sys.executable,
        'dogs_vs_cats/src/sanity_check_random_init.py',
        '--paths-config', PATHS_CFG,
        '--experiment-config', exp['cfg_path'],
    ], check=True)

    sanity_report = Path(PATHS['reports_dir']) / f"sanity_random_init_{exp['experiment_name']}.json"
    if sanity_report.exists():
        sanity = json.loads(sanity_report.read_text())
        print('Sanity passed:', sanity.get('passed'))
        if not sanity.get('passed', False):
            raise RuntimeError(f"Sanity check failed for {exp['experiment_name']}: {sanity_report}")
    else:
        raise FileNotFoundError(f'Missing sanity report: {sanity_report}')


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

TRAIN_RESULTS = []
for exp in EXPERIMENTS:
    print(f"\n[train] {exp['profile_key']} -> {exp['experiment_name']}")
    subprocess.run([
        sys.executable,
        'dogs_vs_cats/src/dinov2_pipeline.py',
        '--mode', 'train',
        '--paths-config', PATHS_CFG,
        '--experiment-config', exp['cfg_path'],
    ], check=True)

    report_path = Path(PATHS['reports_dir']) / f"{exp['experiment_name']}_training_summary.json"
    if not report_path.exists():
        raise FileNotFoundError(f'Missing training summary: {report_path}')

    summary = json.loads(report_path.read_text())
    summary['profile_key'] = exp['profile_key']
    TRAIN_RESULTS.append(summary)

    final_metrics = summary.get('final_val_metrics', {})
    print('Final metrics:', {
        'val_auc': final_metrics.get('val_auc'),
        'val_logloss': final_metrics.get('val_logloss'),
        'val_accuracy': final_metrics.get('val_accuracy'),
    })


In [ ]:
import pandas as pd

rows = []
for summary in TRAIN_RESULTS:
    metrics = summary.get('final_val_metrics', {})
    rows.append(
        {
            'profile': summary.get('profile_key', ''),
            'experiment_name': summary.get('experiment_name', ''),
            'val_auc': metrics.get('val_auc'),
            'val_logloss': metrics.get('val_logloss'),
            'val_accuracy': metrics.get('val_accuracy'),
            'val_loss': metrics.get('val_loss'),
            'monitor': summary.get('monitor'),
            'best_monitor': summary.get('best_monitor'),
            'best_epoch': summary.get('best_epoch'),
        }
    )

compare_df = pd.DataFrame(rows).sort_values(by=['val_auc', 'val_accuracy'], ascending=[False, False]).reset_index(drop=True)
print('Validation comparison (higher AUC/accuracy is better, lower logloss is better):')
display(compare_df)

base_row = compare_df.loc[compare_df['profile'] == 'no_aug']
aug_row = compare_df.loc[compare_df['profile'] == 'strong_aug']
if len(base_row) == 1 and len(aug_row) == 1:
    base_row = base_row.iloc[0]
    aug_row = aug_row.iloc[0]
    deltas = {
        'delta_val_auc': float(aug_row['val_auc'] - base_row['val_auc']),
        'delta_val_logloss': float(aug_row['val_logloss'] - base_row['val_logloss']),
        'delta_val_accuracy': float(aug_row['val_accuracy'] - base_row['val_accuracy']),
    }
    print('Strong_aug - no_aug deltas:', deltas)
else:
    print('Could not compute pairwise deltas (missing profile rows).')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plot_keys = [
    'train_loss_plot',
    'val_metrics_plot',
    'grad_norm_plot',
    'val_confusion_matrix_plot',
]

for summary in TRAIN_RESULTS:
    print('\nPLOTS:', summary['experiment_name'])
    for key in plot_keys:
        path = summary.get('files', {}).get(key)
        if not path:
            continue
        p = Path(path)
        if not p.exists():
            continue
        plt.figure(figsize=(8, 4))
        plt.imshow(mpimg.imread(p))
        plt.title(f"{summary['profile_key']} | {p.name}")
        plt.axis('off')
        plt.show()


In [ ]:
from pathlib import Path

for summary in TRAIN_RESULTS:
    exp_name = summary['experiment_name']
    print('\n' + '=' * 80)
    print('EXPERIMENT:', exp_name)
    print('BEST CHECKPOINT:', summary.get('best_checkpoint'))
    for section, base_dir in [
        ('metrics', PATHS['metrics_dir']),
        ('predictions', PATHS['predictions_dir']),
        ('plots', PATHS['plots_dir']),
        ('reports', PATHS['reports_dir']),
    ]:
        print('\n' + section.upper())
        base = Path(base_dir)
        for item in sorted(base.glob(f'{exp_name}*')):
            print('-', item)
